<a href="https://colab.research.google.com/github/WilsonForeroB/ColabIA/blob/main/FineTuningQLoRA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Actividad: Fine-tuning con QLoRA de un LLM**

**Duración:** 2 horas · **GPU:** T4 gratuita es suficiente (L4/A100 va más rápido)

En la sesión vimos el ciclo **SFT → LoRA → QLoRA**. Hoy lo hacemos de verdad:

> **Problema de negocio:** un equipo de soporte recibe tickets en texto libre. Queremos que un LLM pequeño
> los convierta en un **JSON** con nuestra taxonomía interna (`categoria`, `prioridad`, `producto`, `cliente`)
> para enrutarlos automáticamente.

Un modelo genérico "entiende" el ticket, pero **no conoce nuestras reglas** (qué es prioridad `alta`,
qué nombres de categoría usamos, formato exacto). Eso es justo lo que enseña el fine-tuning:
**el conocimiento es el mismo, solo cambia el comportamiento**.

### Agenda (120 min)
| Min | Bloque | Qué haces |
|---|---|---|
| 0–10 | **Paso 0–1** · Entorno | GPU e instalación |
| 10–20 | **Paso 2** · Repaso | LoRA, QLoRA e hiperparámetros |
| 20–35 | **Paso 3** · Dataset | Generar y entender el formato de entrenamiento |
| 35–50 | **Paso 4–5** · Modelo base + evaluación "antes" | Cargar en 4 bits y medir |
| 50–75 | **Paso 6–7** · Entrenamiento QLoRA | Configurar LoRA y entrenar |
| 75–90 | **Paso 8** · Evaluación "después" | Comparar antes/después |
| 90–105 | **Paso 9** · Guardar y fusionar | Adaptador (MB) vs modelo completo (GB) |
| 105–120 | **Paso 10** · Servir con vLLM + entrega | LoRA en vLLM (opcional) |

> ⏱️ **Si vas justo:** Pasos 0–8 son el núcleo. El 9 y el 10 y el *Reto* son ampliación.

---
## Paso 0 · Verificar GPU
`Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)`

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

## Paso 1 · Instalación
- **transformers**: carga del modelo · **bitsandbytes**: cuantización a 4 bits (la "Q" de QLoRA)
- **peft**: implementación de LoRA · **trl**: `SFTTrainer`, el entrenador de *Supervised Fine-Tuning*

In [ ]:
# ===========================================================================
# PASO 1: Instalación (~1-2 min)
# ===========================================================================
!pip install -q -U transformers trl peft bitsandbytes accelerate datasets

import torch, transformers, trl, peft
print("torch", torch.__version__, "| transformers", transformers.__version__, "| trl", trl.__version__, "| peft", peft.__version__)
BF16 = torch.cuda.get_device_capability() >= (8, 0)   # T4 → fp16, L4/A100 → bf16
DTYPE = torch.bfloat16 if BF16 else torch.float16
print("GPU:", torch.cuda.get_device_name(), "| dtype de cálculo:", DTYPE)

---
## Paso 2 · Repaso en 10 minutos

```
Full fine-tuning:   W (1.500 M parámetros)  ← se actualizan TODOS           → mucha VRAM
LoRA:               W' = W + (α/r)·A·B       W congelado, solo A y B         → ~0,5–2% de parámetros
QLoRA:              W en 4 bits (NF4) + A·B en 16 bits                       → cabe en una T4
```

### Hiperparámetros que vas a tocar
| Parámetro | Valor hoy | Qué controla | Si lo subes... |
|---|---|---|---|
| `r` (rango) | 16 | Tamaño de las matrices A y B | Más capacidad, más memoria |
| `lora_alpha` | 32 | Escala de la actualización (regla: 2·r) | Cambios más fuertes |
| `target_modules` | `all-linear` | En qué capas se insertan A·B | Más capas = mejor adaptación |
| `learning_rate` | 2e-4 | Tamaño de cada paso | Aprende rápido pero puede romperse |
| `num_train_epochs` | 2 | Veces que ve el dataset | Riesgo de **overfitting** |
| Nº de ejemplos | 300 | Datos de entrenamiento | Pocos → **underfitting** |

### Recordatorio de la sesión: ¿cuántos datos?
**Muy pocos** → no aprende (*underfitting*) · **Suficientes** → aprende bien · **Repetir demasiado** → memoriza (*overfitting*).
**La calidad de los datos importa más que la cantidad.**

---
## Paso 3 · Dataset

En un proyecto real exportarías tickets históricos ya etiquetados. Aquí los **generamos sintéticamente**
para que todo el mundo tenga los mismos datos.

**Detalle importante:** las frases del conjunto de *test* incluyen una plantilla por categoría que el modelo **nunca ve en entrenamiento**.
Así comprobamos que **generaliza** y no solo memoriza.

In [ ]:
# ===========================================================================
# PASO 3a: Generador de tickets sintéticos
# ===========================================================================
import random, json

PRODUCTOS = ["Router X200", "Smartwatch Fit3", "Plan Fibra 600", "Portátil Nova 14",
             "Cafetera BrewPro", "Altavoz Sonix", "Tablet Lumo 10", "Impresora Inkjet P5"]
NOMBRES = ["Ana", "Luis", "Marta", "Carlos", "Lucía", "Javier", "Sofía", "Pablo", "Elena", "Diego", "Carmen", "Raúl"]
APELLIDOS = ["García", "López", "Martínez", "Sánchez", "Pérez", "Gómez", "Ruiz", "Díaz", "Moreno", "Navarro"]

# La ÚLTIMA plantilla de cada categoría solo aparece en test (frases nunca vistas)
PROBLEMAS = {
    "facturacion": ["me habéis cobrado dos veces el {p} este mes",
                    "la factura del {p} tiene un importe que no cuadra",
                    "quiero que me devolváis el dinero del {p}",
                    "no me ha llegado la factura de mi {p}",
                    "en el extracto del banco aparece un cargo del {p} que no reconozco"],
    "soporte_tecnico": ["el {p} se queda colgado al arrancar",
                        "no consigo conectar el {p} a la wifi",
                        "el {p} muestra el error E-{n} desde ayer",
                        "la app del {p} se cierra sola",
                        "desde la última actualización el {p} va lentísimo y se reinicia"],
    "envio": ["mi pedido del {p} lleva {n} días sin llegar",
              "me ha llegado el {p} con la caja destrozada",
              "el seguimiento del {p} no se actualiza",
              "he recibido un {p} que no pedí",
              "el repartidor dice que entregó el {p} pero yo no tengo nada"],
    "cuenta": ["no puedo iniciar sesión para gestionar mi {p}",
               "quiero cambiar el email asociado a mi {p}",
               "me han bloqueado la cuenta después de comprar el {p}",
               "quiero darme de baja del servicio del {p}",
               "alguien ha cambiado la contraseña de mi cuenta del {p} sin mi permiso"],
}
PRIORIDAD = {
    "alta": ["Es urgente, lo necesito hoy.", "Mi negocio está parado por esto.",
             "Necesito una solución YA, es inaceptable.", "Por favor, prioridad máxima."],
    "media": ["¿Podéis revisarlo esta semana?", "Agradecería una respuesta pronto.",
              "Me gustaría resolverlo en los próximos días.", "Espero noticias vuestras en breve."],
    "baja": ["No corre prisa.", "Cuando podáis, sin urgencia.",
             "Solo quería que lo supierais.", "Sin prisa, es por si os sirve."],
}
SALUDOS = ["Hola, soy {c}.", "Buenas, me llamo {c}.", "Buenos días, os escribe {c}.", ""]
DESPEDIDAS = ["Gracias.", "Un saludo, {c}", "Atentamente, {c}", "Saludos."]

def generar_ticket(rng, test=False):
    cat = rng.choice(list(PROBLEMAS))
    pri = rng.choice(list(PRIORIDAD))
    prod = rng.choice(PRODUCTOS)
    cli = f"{rng.choice(NOMBRES)} {rng.choice(APELLIDOS)}"
    plantillas = PROBLEMAS[cat] if test else PROBLEMAS[cat][:-1]
    problema = rng.choice(plantillas).format(p=prod, n=rng.randint(2, 999))
    saludo = rng.choice(SALUDOS)
    despedida = rng.choice(DESPEDIDAS)
    if "{c}" not in saludo + despedida:              # el nombre siempre debe aparecer
        saludo = "Hola, soy {c}."
    texto = " ".join(x for x in [saludo.format(c=cli), problema[0].upper() + problema[1:] + ".",
                                 rng.choice(PRIORIDAD[pri]), despedida.format(c=cli)] if x)
    etiqueta = {"categoria": cat, "prioridad": pri, "producto": prod, "cliente": cli}
    return texto, etiqueta

rng = random.Random(0)
for _ in range(3):
    t, e = generar_ticket(rng)
    print("📨", t, "\n🏷️", json.dumps(e, ensure_ascii=False), "\n")

### Paso 3b · Formato de entrenamiento: *prompt → completion*

Cada ejemplo es una conversación: el **prompt** (instrucciones + ticket) y la **completion** (el JSON correcto).
Usando este formato, `SFTTrainer` calcula la pérdida **solo sobre la respuesta**: no queremos que el modelo
aprenda a escribir tickets, sino a **responderlos**.

In [ ]:
# ===========================================================================
# PASO 3b: Construir datasets de entrenamiento y test
# ===========================================================================
from datasets import Dataset

SYSTEM = ("Eres un clasificador de tickets de soporte. Devuelve SOLO un JSON con las claves "
          "categoria (facturacion | soporte_tecnico | envio | cuenta), "
          "prioridad (alta | media | baja), producto y cliente.")

def a_ejemplo(texto, etiqueta):
    return {
        "prompt": [{"role": "system", "content": SYSTEM},
                   {"role": "user", "content": texto}],
        "completion": [{"role": "assistant", "content": json.dumps(etiqueta, ensure_ascii=False)}],
    }

N_TRAIN, N_TEST = 300, 60
rng_train, rng_test = random.Random(42), random.Random(2024)
train_ds = Dataset.from_list([a_ejemplo(*generar_ticket(rng_train)) for _ in range(N_TRAIN)])
test_raw = [generar_ticket(rng_test, test=True) for _ in range(N_TEST)]

print(train_ds)
print(json.dumps(train_ds[0], ensure_ascii=False, indent=1))

---
## Paso 4 · Cargar el modelo base en 4 bits (la "Q" de QLoRA)

`BitsAndBytesConfig` comprime los pesos a **NF4** (4 bits, *NormalFloat*, pensado para pesos con distribución normal).
Con *double quant* también se cuantizan las escalas → ahorro extra.

In [ ]:
# ===========================================================================
# PASO 4: Modelo base cuantizado a 4 bits
# ===========================================================================
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"   # más rápido: "Qwen/Qwen2.5-0.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=DTYPE,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config,
                                             device_map="auto", torch_dtype=DTYPE)
print(f"VRAM usada por el modelo en 4 bits: {torch.cuda.memory_allocated()/2**30:.2f} GiB")
print("(el mismo modelo en FP16 ocupa ~3 GiB)")

---
## Paso 5 · Evaluación "ANTES" del fine-tuning

Métricas sencillas y objetivas:
- **JSON válido**: ¿se puede parsear la salida?
- **Acierto por campo**: ¿coincide exactamente con la etiqueta?
- **Todo correcto**: los 4 campos bien → el ticket se enruta sin intervención humana.

In [ ]:
# ===========================================================================
# PASO 5: Función de evaluación
# ===========================================================================
import re, pandas as pd

@torch.no_grad()
def generar(model, textos, batch_size=16, max_new_tokens=80):
    tokenizer.padding_side = "left"
    salidas = []
    for i in range(0, len(textos), batch_size):
        prompts = [tokenizer.apply_chat_template(
                       [{"role": "system", "content": SYSTEM}, {"role": "user", "content": t}],
                       tokenize=False, add_generation_prompt=True)
                   for t in textos[i:i + batch_size]]
        enc = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)
        out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, use_cache=True,
                             pad_token_id=tokenizer.pad_token_id)
        salidas += tokenizer.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    tokenizer.padding_side = "right"
    return salidas

def evaluar(model, nombre):
    textos = [t for t, _ in test_raw]
    preds = generar(model, textos)
    filas = []
    for (texto, gold), pred in zip(test_raw, preds):
        m = re.search(r"\{.*\}", pred, re.S)
        try:
            p = json.loads(m.group(0)) if m else None
        except json.JSONDecodeError:
            p = None
        fila = {"json_valido": p is not None}
        for k in gold:
            fila[k] = bool(p) and str(p.get(k, "")).strip().lower() == gold[k].lower()
        fila["todo_ok"] = all(fila[k] for k in gold)
        filas.append(fila)
    df = pd.DataFrame(filas)
    res = (df.mean() * 100).round(1).rename(nombre)
    return res, preds

res_antes, preds_antes = evaluar(model, "ANTES")
print(res_antes.to_string(), "\n")
for (t, g), p in list(zip(test_raw, preds_antes))[:3]:
    print("📨", t, "\n✅", json.dumps(g, ensure_ascii=False), "\n🤖", p.strip(), "\n")

> 🔎 **Observa:** ¿el modelo base inventa categorías ("Facturación", "técnico", "Envíos")? ¿Añade texto fuera del JSON?
> ¿Decide la prioridad con otro criterio? Eso es lo que vamos a corregir.

---
## Paso 6 · Configurar LoRA

Insertamos las matrices **A·B** en todas las capas lineales. Fíjate en el porcentaje de parámetros entrenables.

In [ ]:
# ===========================================================================
# PASO 6: Configuración LoRA
# ===========================================================================
from peft import LoraConfig

lora_config = LoraConfig(
    r=16,                         # rango de A·B
    lora_alpha=32,                # escala (regla práctica: 2·r)
    lora_dropout=0.05,
    target_modules="all-linear",  # q,k,v,o + MLP (gate, up, down)
    task_type="CAUSAL_LM",
)

## Paso 7 · Entrenar con `SFTTrainer` (≈5–10 min en T4)

Mira la columna **`loss`**: debe bajar rápido al principio y estabilizarse.
Si llega casi a 0 muy pronto, sospecha de *overfitting*.

In [ ]:
# ===========================================================================
# PASO 7: Entrenamiento QLoRA
# ===========================================================================
from trl import SFTTrainer, SFTConfig

args = SFTConfig(
    output_dir="/content/qlora-tickets",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,     # batch efectivo = 16
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    logging_steps=5,
    max_length=512,
    bf16=BF16, fp16=not BF16,
    gradient_checkpointing=True,       # menos VRAM a cambio de algo de velocidad
    save_strategy="no",
    report_to="none",
)

trainer = SFTTrainer(model=model, args=args, train_dataset=train_ds,
                     peft_config=lora_config, processing_class=tokenizer)
trainer.model.print_trainable_parameters()   # 👀 ¿qué % se entrena?

stats = trainer.train()
print(f"\n⏱️ {stats.metrics['train_runtime']:.0f} s | VRAM pico: {torch.cuda.max_memory_allocated()/2**30:.2f} GiB")

In [ ]:
# Curva de pérdida
import matplotlib.pyplot as plt
log = pd.DataFrame([x for x in trainer.state.log_history if "loss" in x])
plt.plot(log["step"], log["loss"], marker="o"); plt.xlabel("step"); plt.ylabel("loss")
plt.title("Pérdida de entrenamiento"); plt.grid(alpha=.3); plt.show()

---
## Paso 8 · Evaluación "DESPUÉS"

In [ ]:
# ===========================================================================
# PASO 8: Comparar antes / después
# ===========================================================================
trainer.model.eval()
res_despues, preds_despues = evaluar(trainer.model, "DESPUÉS")
display(pd.concat([res_antes, res_despues], axis=1))

print("\nEjemplos con frases NUNCA vistas en entrenamiento:")
nuevas = [i for i, (t, _) in enumerate(test_raw)
          if any(PROBLEMAS[c][-1].split("{")[0][:25].lower() in t.lower() for c in PROBLEMAS)][:4]
for i in nuevas:
    print("📨", test_raw[i][0], "\n✅", json.dumps(test_raw[i][1], ensure_ascii=False),
          "\n🤖", preds_despues[i].strip(), "\n")

> 💬 **Discute:** ¿qué campo mejoró más? ¿Falla algo en las frases nuevas? ¿Por qué el campo `cliente`
> probablemente ya iba bien antes? (pista: es extracción literal, no una regla de negocio).

---
## 🧪 Reto opcional (15 min): underfitting vs overfitting
Repite los Pasos 4, 6, 7 y 8 cambiando **una sola cosa** y anota `todo_ok`:

| Experimento | Cambio | Hipótesis |
|---|---|---|
| Pocos datos | `N_TRAIN = 20` | *Underfitting*: no aprende la taxonomía |
| Muchas épocas | `num_train_epochs = 10` con `N_TRAIN = 50` | *Overfitting*: loss ≈ 0, falla en frases nuevas |
| Rango mínimo | `r = 2, lora_alpha = 4` | ¿Basta con tan poca capacidad para esta tarea? |
| LR alta | `learning_rate = 2e-3` | Entrenamiento inestable |

> Antes de repetir, libera memoria: `del trainer, model; import gc; gc.collect(); torch.cuda.empty_cache()`

---
## Paso 9 · Guardar el adaptador y fusionarlo

Lo que se guarda tras QLoRA **no es el modelo**, son solo las matrices A y B (**decenas de MB**).
Para desplegar tienes dos opciones:
1. **Servir base + adaptador** (vLLM lo soporta; puedes tener muchos adaptadores sobre un mismo modelo base).
2. **Fusionar** `W' = W + A·B` en un modelo completo → luego lo puedes **cuantizar** con la actividad `CuantizacionVLLM.ipynb`.

In [ ]:
# ===========================================================================
# PASO 9a: Guardar solo el adaptador LoRA
# ===========================================================================
ADAPTER_DIR = "/content/lora-tickets"
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
!du -sh $ADAPTER_DIR
!ls -lh $ADAPTER_DIR

In [ ]:
# ===========================================================================
# PASO 9b: Fusionar con el modelo base en 16 bits
# ===========================================================================
# Se fusiona sobre el modelo ORIGINAL en 16 bits (no sobre el de 4 bits) para no arrastrar error de cuantización.
from peft import PeftModel
import gc
del trainer, model; gc.collect(); torch.cuda.empty_cache()

base = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=DTYPE, device_map="auto")
merged = PeftModel.from_pretrained(base, ADAPTER_DIR).merge_and_unload()

MERGED_DIR = "/content/qwen-tickets-merged"
merged.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)
!du -sh $MERGED_DIR

# Comprobación rápida: el modelo fusionado debe dar el mismo resultado
res_merged, _ = evaluar(merged, "FUSIONADO_16bit")
display(pd.concat([res_antes, res_despues, res_merged], axis=1))
del merged, base; gc.collect(); torch.cuda.empty_cache()

---
## Paso 10 · (Opcional) Servir el adaptador LoRA con vLLM

Conexión con la actividad de cuantización: vLLM puede cargar el **modelo base una vez** y aplicar
adaptadores LoRA **por petición** (útil si tienes un adaptador por cliente o por tarea).

> ⚠️ Instalar vLLM cambia la versión de PyTorch: hazlo **al final** y **reinicia la sesión** después.
> Los archivos en `/content` se conservan al reiniciar la sesión.

In [ ]:
!pip install -q uv && uv pip install --system -q vllm
print("👉 Entorno de ejecución → Reiniciar sesión, y ejecuta la siguiente celda")

In [ ]:
%%writefile /content/serve_lora.py
import json, torch
from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest

SYSTEM = ("Eres un clasificador de tickets de soporte. Devuelve SOLO un JSON con las claves "
          "categoria (facturacion | soporte_tecnico | envio | cuenta), "
          "prioridad (alta | media | baja), producto y cliente.")
ticket = "Buenas, me llamo Irene Castro. El Router X200 parpadea en rojo y no hay internet. Mi negocio está parado por esto."
msgs = [[{"role": "system", "content": SYSTEM}, {"role": "user", "content": ticket}]]

kw = dict(model="Qwen/Qwen2.5-1.5B-Instruct", enable_lora=True, max_lora_rank=16,
          max_model_len=1024, gpu_memory_utilization=0.8, enforce_eager=True)
if torch.cuda.get_device_capability() < (8, 0):
    kw["dtype"] = "half"
llm = LLM(**kw)
sp = SamplingParams(temperature=0, max_tokens=80)

print("\n🧊 BASE   :", llm.chat(msgs, sp)[0].outputs[0].text)
print("🔥 + LoRA :", llm.chat(msgs, sp, lora_request=LoRARequest("tickets", 1, "/content/lora-tickets"))[0].outputs[0].text)

In [ ]:
!python /content/serve_lora.py 2>&1 | grep -E "🧊|🔥|Traceback|Error"

Como servidor con API compatible con OpenAI (el adaptador aparece como un modelo más llamado `tickets`):
```bash
vllm serve Qwen/Qwen2.5-1.5B-Instruct --enable-lora --max-lora-rank 16 \
     --lora-modules tickets=/content/lora-tickets
```
Y en el cliente: `client.chat.completions.create(model="tickets", messages=[...])`.

---
## 🧰 Otras herramientas para hacer lo mismo
| Herramienta | Para qué | Cuándo |
|---|---|---|
| **TRL + PEFT** (esta actividad) | Control total, código estándar de Hugging Face | Aprender y proyectos a medida |
| **Unsloth** | Mismo flujo, ~2× más rápido y menos VRAM; notebooks listos en Colab | Entrenar modelos más grandes en GPU modesta |
| **LLaMA-Factory** | Interfaz web (*LLaMA Board*) y configuración YAML, sin código | Equipos no técnicos o muchas pruebas |
| **DPO (TRL `DPOTrainer`)** | Alinear con pares respuesta buena/mala | Después del SFT, para pulir estilo y preferencias |

## 🆘 Problemas frecuentes
| Error | Solución |
|---|---|
| `CUDA out of memory` | Baja `per_device_train_batch_size` a 4 (y sube `gradient_accumulation_steps` a 4), o usa el modelo 0.5B |
| `bf16 ... not supported` | Estás en T4: el notebook ya usa fp16 automáticamente (`BF16=False`) |
| `SFTConfig got an unexpected keyword 'max_length'` | TRL antiguo: usa `max_seq_length=512` o actualiza `trl` |
| Loss en `nan` | Baja `learning_rate` a 1e-4 |
| Tras instalar vLLM falla `trl`/`peft` | Normal, cambió PyTorch: el Paso 10 es la última parte de la actividad |

---
## 📝 Entrega
1. Captura de la tabla **ANTES / DESPUÉS** y de la curva de pérdida.
2. Responde en 5–6 líneas:
   - ¿Qué % de parámetros entrenaste y cuánto ocupa el adaptador frente al modelo fusionado?
   - ¿Qué campo mejoró más y por qué crees que es así?
   - ¿El modelo generaliza a las frases nunca vistas?
   - (Si hiciste el reto) ¿Observaste underfitting u overfitting? ¿Con qué configuración?
   - ¿Cuándo usarías fine-tuning y cuándo bastaría con un buen prompt o con RAG?